# imports

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from neuralforecast import NeuralForecast
import optuna
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd
from pathlib import Path
import pathlib
import math
import holidays
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os
import random
import torch
import warnings
import xgboost as xgb
import lightgbm as lgb
import torch
torch.set_float32_matmul_precision("medium")
from neuralforecast.models import GRU
from sklearn.metrics import root_mean_squared_error
from neuralforecast.losses.pytorch import MSE
from scipy.stats import gaussian_kde, norm
from sklearn.ensemble import RandomForestRegressor
import re


# ============================================================
# GLOBAL SETTINGS FOR LIGHTWEIGHT HPO
# ============================================================
MAX_STEPS = 500
VAL_CHECK_STEPS = MAX_STEPS // 10




def get_holiday_calendar(country):
    if country == "Germany":
        return holidays.Germany()
    elif country == "Ireland":
        return holidays.Ireland()
    elif country == "Portugal":
        return holidays.Portugal()
    elif country == "Denmark":
        return holidays.Denmark()
    else:
        return holidays.Germany()


def add_calendar_features(df, country):
    df = df.copy()

    df["minute"] = df["ds"].dt.minute
    df["hour"] = df["ds"].dt.hour
    df["day_of_week"] = df["ds"].dt.dayofweek
    df["day_of_year"] = df["ds"].dt.dayofyear
    df["week"] = df["ds"].dt.isocalendar().week.astype(int)
    df["month"] = df["ds"].dt.month
    df["year"] = df["ds"].dt.year
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    holiday_calendar = get_holiday_calendar(country)
    df["holiday"] = df["ds"].dt.normalize().map(
        lambda x: 1 if x in holiday_calendar else 0
    )

    minute_period = 60
    hour_period = 24
    week_period = 7
    weeks_inyear= 52
    month_period = 12
    year_period = 365.25

    df["minute_sin"] = np.sin(2 * np.pi * df["minute"] / minute_period)
    df["minute_cos"] = np.cos(2 * np.pi * df["minute"] / minute_period)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / hour_period)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / hour_period)
    df["dayofweek_sin"] = np.sin(2 * np.pi * df["day_of_week"] / week_period)
    df["dayofweek_cos"] = np.cos(2 * np.pi * df["day_of_week"] / week_period)
    df["dayofyear_sin"] = np.sin(2 * np.pi * df["day_of_year"] / year_period)
    df["dayofyear_cos"] = np.cos(2 * np.pi * df["day_of_year"] / year_period)
    df["week_sin"] = np.sin(2 * np.pi * df["week"] / weeks_inyear)
    df["week_cos"] = np.cos(2 * np.pi * df["week"] / weeks_inyear)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / month_period)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / month_period)

    return df


def select_top_correlated_weather_lags(
    train_df,
    weather_cols,
    forecast_horizon,
    top_k_per_weather=6,
    max_weather_lag=None,
):
    if max_weather_lag is None:
        max_weather_lag = forecast_horizon * 2

    df = train_df.copy().sort_values(["unique_id", "ds"]).reset_index(drop=True)

    selected_weather_lag_features = []

    for col in weather_cols:
        corr_rows = []
        g = df.groupby("unique_id")[col]

        for lag in range(forecast_horizon, max_weather_lag + 1):
            lag_name = f"{col}_lag_{lag}"
            lag_values = g.shift(lag)

            tmp = pd.DataFrame({
                "y": df["y"].values,
                lag_name: lag_values.values
            }).dropna()

            if len(tmp) < 10:
                corr = 0.0
            else:
                corr = tmp["y"].corr(tmp[lag_name])
                if pd.isna(corr):
                    corr = 0.0

            corr_rows.append({
                "feature": lag_name,
                "abs_corr": abs(corr),
            })

        corr_df = pd.DataFrame(corr_rows).sort_values("abs_corr", ascending=False)
        best_feats = corr_df.head(top_k_per_weather)["feature"].tolist()
        selected_weather_lag_features.extend(best_feats)

    return sorted(selected_weather_lag_features)


def add_selected_weather_lags(df, selected_weather_lag_features):
    df = df.copy().sort_values(["unique_id", "ds"]).reset_index(drop=True)

    for feat in selected_weather_lag_features:
        m = re.fullmatch(r"(.+)_lag_(\d+)", feat)
        if m:
            base_col = m.group(1)
            lag = int(m.group(2))
            df[feat] = df.groupby("unique_id")[base_col].shift(lag)

    return df


def get_feature_groups():
    future_known_features = [
        "minute", "hour", "day_of_week", "day_of_year", "week", "month", "year",
        "is_weekend", "holiday",
        "minute_sin", "minute_cos",
        "hour_sin", "hour_cos",
        "dayofweek_sin", "dayofweek_cos",
        "dayofyear_sin", "dayofyear_cos",
        "week_sin", "week_cos",
        "month_sin", "month_cos",
    ]
    return future_known_features


def build_feature_enriched_df(
    df_all,
    home_cols,
    weather_cols,
    country,
    forecast_horizon,
    top_k_per_weather=6,
    max_weather_lag=None,
    train_end_for_selection=None,
):
    if max_weather_lag is None:
        max_weather_lag = forecast_horizon * 2

    df_nf = build_local_nf_df(df_all, home_cols, weather_cols)
    df_nf = add_calendar_features(df_nf, country=country)

    if train_end_for_selection is None:
        raise ValueError("train_end_for_selection must be provided.")

    train_only_df = df_nf[df_nf["ds"] < pd.Timestamp(train_end_for_selection)].copy()

    selected_weather_lag_features = select_top_correlated_weather_lags(
        train_df=train_only_df,
        weather_cols=weather_cols,
        forecast_horizon=forecast_horizon,
        top_k_per_weather=top_k_per_weather,
        max_weather_lag=max_weather_lag,
    )

    df_nf = add_selected_weather_lags(
        df=df_nf,
        selected_weather_lag_features=selected_weather_lag_features,
    )

    return df_nf, selected_weather_lag_features


def _mad(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return np.median(np.abs(x - med))


def empirical_bayes_threshold_from_importance(
    importances,
    feature_names,
    alpha=0.20,
    transform="log1p",
    central_prop=0.80,
    random_state=42,
):
    imp = np.asarray(importances, dtype=float)
    if np.any(imp < 0):
        raise ValueError("Importances must be non-negative.")

    names = np.asarray(feature_names)
    if len(names) != len(imp):
        raise ValueError("feature_names and importances must have the same length.")

    rng = np.random.default_rng(random_state)
    eps = 1e-12
    imp_j = imp + eps * rng.normal(size=len(imp))

    if transform == "log1p":
        z = np.log1p(np.maximum(imp_j, 0.0))
    elif transform is None:
        z = imp_j.copy()
    else:
        raise ValueError("transform must be 'log1p' or None")

    q_low = (1.0 - central_prop) / 2.0
    q_high = 1.0 - q_low
    lo, hi = np.quantile(z, [q_low, q_high])
    z_central = z[(z >= lo) & (z <= hi)]

    mu0 = np.median(z_central)
    sigma0 = 1.4826 * _mad(z_central)
    sigma0 = max(sigma0, 1e-6)

    if len(np.unique(z)) < 2:
        f_z = np.ones_like(z)
    else:
        kde = gaussian_kde(z)
        f_z = kde.evaluate(z)

    f0_z = norm.pdf(z, loc=mu0, scale=sigma0)

    pi0 = np.mean(z <= mu0 + sigma0)
    pi0 = float(np.clip(pi0, 0.50, 0.99))

    local_fdr = np.clip(pi0 * f0_z / np.maximum(f_z, 1e-12), 0.0, 1.0)
    selected = local_fdr <= alpha

    results_df = pd.DataFrame({
        "feature": names,
        "importance_raw": imp,
        "importance_transformed": z,
        "local_fdr": local_fdr,
        "selected": selected,
    }).sort_values(
        by=["selected", "importance_raw", "local_fdr"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    if results_df["selected"].any():
        threshold_raw = results_df.loc[results_df["selected"], "importance_raw"].min()
    else:
        threshold_raw = np.inf

    return results_df, threshold_raw


def select_exogenous_features(
    train_df,
    future_known_features,
    historical_lagged_features,
    alpha=0.20,
    rf_params=None,
    fallback_top_k=25,
):
    candidate_features = future_known_features + historical_lagged_features

    feat_df = train_df[["unique_id", "ds", "y"] + candidate_features].copy()
    feat_df = feat_df.dropna().reset_index(drop=True)

    X = feat_df[candidate_features].copy()
    y = feat_df["y"].values

    if rf_params is None:
        rf_params = {
            "n_estimators": 500,
            "random_state": 42,
            "n_jobs": -1,
            "max_features": "sqrt",
        }

    rf = RandomForestRegressor(**rf_params)
    rf.fit(X, y)

    importance_df, threshold_raw = empirical_bayes_threshold_from_importance(
        importances=rf.feature_importances_,
        feature_names=X.columns.tolist(),
        alpha=alpha,
        transform="log1p",
        central_prop=0.80,
        random_state=42,
    )

    selected_features = importance_df.loc[importance_df["selected"], "feature"].tolist()

    if len(selected_features) == 0:
        selected_features = (
            importance_df.sort_values("importance_raw", ascending=False)
            .head(min(fallback_top_k, len(importance_df)))["feature"]
            .tolist()
        )
        importance_df["selected"] = importance_df["feature"].isin(selected_features)
        threshold_raw = importance_df.loc[
            importance_df["feature"].isin(selected_features), "importance_raw"
        ].min()

    selected_future_known = [f for f in selected_features if f in future_known_features]
    selected_historical_lags = [f for f in selected_features if f in historical_lagged_features]

    print(f"Empirical-Bayes threshold (raw importance): {threshold_raw:.8f}")
    print(f"Selected future-known exogenous features: {len(selected_future_known)}")
    print(f"Selected historical lagged exogenous features: {len(selected_historical_lags)}")

    return selected_future_known, selected_historical_lags, importance_df


def keep_only_required_columns(df, selected_future_known, selected_historical_lags):
    keep_cols = ["unique_id", "ds", "y"] + selected_future_known + selected_historical_lags
    keep_cols = [c for c in keep_cols if c in df.columns]
    return df[keep_cols].copy()

def split_train_val_test_local(df_nf, test_start, forecast_horizon, training_size, val_days=3, freq_minutes=15):
    """
    Split a long NeuralForecast dataframe into train / validation / test per series.

    Validation starts `val_days` before test_start and ends right before test_start.
    Test spans `forecast_horizon` steps starting at test_start.
    Training is the last `training_size` rows before validation starts.
    """
    test_start = pd.Timestamp(test_start)
    val_start = test_start - pd.Timedelta(days=val_days)
    test_end = test_start + pd.Timedelta(minutes=freq_minutes * forecast_horizon)

    train_parts = []
    val_parts = []
    test_parts = []

    expected_val_len = val_days * (24 * 60 // freq_minutes)

    for uid, g in df_nf.groupby("unique_id"):
        g = g.sort_values("ds").reset_index(drop=True)

        # train: everything before validation starts, keep only last training_size rows
        train_candidates = g[g["ds"] < val_start].copy()
        train_df_uid = train_candidates.iloc[-training_size:].copy()

        # validation: from val_start until just before test_start
        val_df_uid = g[(g["ds"] >= val_start) & (g["ds"] < test_start)].copy()

        # test: from test_start for forecast_horizon steps
        test_df_uid = g[(g["ds"] >= test_start) & (g["ds"] < test_end)].copy()

        # safety checks
        if len(train_df_uid) != training_size:
            raise ValueError(f"{uid}: expected {training_size} training rows, got {len(train_df_uid)}")

        if len(val_df_uid) != expected_val_len:
            raise ValueError(f"{uid}: expected {expected_val_len} validation rows, got {len(val_df_uid)}")

        if len(test_df_uid) != forecast_horizon:
            raise ValueError(f"{uid}: expected {forecast_horizon} test rows, got {len(test_df_uid)}")

        train_parts.append(train_df_uid)
        val_parts.append(val_df_uid)
        test_parts.append(test_df_uid)

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, val_df, test_df


def build_local_nf_df(df_all, home_cols, weather_cols=None):
    """
    Convert a wide household load dataframe into NeuralForecast long format.

    Parameters
    ----------
    df_all : pd.DataFrame
        Index must be DatetimeIndex, columns include home_* and optional weather cols.
    home_cols : list
        List of household columns, e.g. ['home_1', 'home_2', ...]
    weather_cols : list or None
        Optional list of weather columns to attach to every home/timestamp row.

    Returns
    -------
    df_nf : pd.DataFrame
        Columns: unique_id, ds, y, [weather columns...]
    """
    if not isinstance(df_all.index, pd.DatetimeIndex):
        raise ValueError("df_all index must be a DatetimeIndex.")

    # keep time as a normal column
    df_base = df_all.reset_index().rename(columns={"timestamp": "ds"})

    # wide -> long for homes
    df_nf = df_base.melt(
        id_vars=["ds"],
        value_vars=home_cols,
        var_name="unique_id",
        value_name="y"
    )

    # add weather columns if requested
    if weather_cols is not None and len(weather_cols) > 0:
        weather_df = df_base[["ds"] + weather_cols].copy()
        df_nf = df_nf.merge(weather_df, on="ds", how="left")

    # sort for safety
    df_nf = df_nf.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    return df_nf



def rolling_forecasting_validation_predictions_local(
    train_df,
    val_df,
    h,
    model_params,
    future_known_features,
    historical_lagged_features,
    freq="15min"
):
    rolling_train_df = train_df.copy().sort_values(["unique_id", "ds"]).reset_index(drop=True)
    val_predictions = []

    val_starts = sorted(val_df["ds"].unique())[::h]

    for window_i, window_start in enumerate(val_starts, start=1):
        print(f"\nROLLING WINDOW {window_i}/{len(val_starts)} | window_start={window_start}")

        model = GRU(
            h=h,
            input_size=model_params["input_size"],
            encoder_n_layers=model_params["encoder_n_layers"],
            encoder_hidden_size=model_params["encoder_hidden_size"],
            encoder_dropout=model_params["encoder_dropout"],
            decoder_layers=model_params["decoder_layers"],
            decoder_hidden_size=model_params["decoder_hidden_size"],
            batch_size=model_params["batch_size"],
            learning_rate=model_params["learning_rate"],
            max_steps=MAX_STEPS,
            val_check_steps=min(VAL_CHECK_STEPS, MAX_STEPS),
            scaler_type="standard",
            random_seed=42,
            loss=MSE(),
            futr_exog_list=future_known_features,
            hist_exog_list=historical_lagged_features,
        )

        nf = NeuralForecast(models=[model], freq=freq)

        nf.fit(df=rolling_train_df)

        future_chunk = val_df[
            (val_df["ds"] >= window_start) &
            (val_df["ds"] < window_start + pd.Timedelta(minutes=15 * h))
        ].copy()

        futr_df = future_chunk[["unique_id", "ds"] + future_known_features].copy()

        preds = nf.predict(futr_df=futr_df)
        val_predictions.append(preds)

        rolling_train_df = pd.concat([rolling_train_df, future_chunk], ignore_index=True)
        rolling_train_df = rolling_train_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    if len(val_predictions) == 0:
        raise ValueError("No validation predictions were generated.")

    return pd.concat(val_predictions, ignore_index=True)

def compute_rmse_local(val_df, val_preds_df, pred_col="GRU"):
    val_compare_df = val_df.merge(val_preds_df, on=["unique_id", "ds"], how="left")

    rmse_value = root_mean_squared_error(
        val_compare_df["y"],
        val_compare_df[pred_col]
    )

    return rmse_value, val_compare_df



def make_objective_local(
    train_df,
    val_df,
    forecast_horizon,
    selected_future_known,
    selected_historical_lags,
    freq="15min"
):
    def objective(trial):

        model_params = {
            "input_size": trial.suggest_categorical("input_size", [96, 192, 288, 672]),
            "encoder_n_layers": trial.suggest_int("encoder_n_layers", 1, 3, step=1),
            "encoder_hidden_size": trial.suggest_int("encoder_hidden_size", 50, 250, step=50),
            "encoder_dropout": trial.suggest_float("encoder_dropout", 0.0, 0.3),
            "decoder_layers": trial.suggest_int("decoder_layers", 1, 3, step=1),
            "decoder_hidden_size": trial.suggest_int("decoder_hidden_size", 50, 250, step=50),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        }

        try:
            val_preds_df = rolling_forecasting_validation_predictions_local(
                train_df=train_df,
                val_df=val_df,
                h=forecast_horizon,
                model_params=model_params,
                future_known_features=selected_future_known,
                historical_lagged_features=selected_historical_lags,
                freq=freq,
            )

            rmse_value, _ = compute_rmse_local(
                val_df=val_df,
                val_preds_df=val_preds_df,
                pred_col="GRU"
            )

            return rmse_value

        except Exception as e:
            import traceback
            print(f"Trial failed: {e}")
            traceback.print_exc()
            return float("inf")

    return objective

# start

In [ ]:
import time

start_time = time.time()

project_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone"
days_json_path = pathlib.Path(project_path) / "dataset_days.json"

#countries = ["Germany", "Ireland", "Portugal"]
#countries = ["Germany"]
countries = ["Germany", "Ireland", "Portugal","Denmark"]

#days = ["day1", "day2", "day3", "day4", "day5"]
days = ["day1"]



weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
#    "price_eur_kwh"
]


forecast_horizon = 96
training_size = 96 * 7 * 3 * 2
opt_trials = 10

with open(days_json_path, "r") as f:
    dataset_days = json.load(f)


# -------------------------
# Loop over countries
# -------------------------
for country in countries:
    country_start_time = time.time()

    print(f"\n{'#'*100}")
    print(f"COUNTRY: {country}")
    print(f"{'#'*100}")

    if country == "Denmark" and "price_eur_kwh" not in weather_cols:

        weather_cols.append("price_eur_kwh")

    dataset_path = pathlib.Path(project_path) / "DataCleaning" / "clean" / f"dataset_{country}.csv"

    df_all = pd.read_csv(dataset_path, parse_dates=["timestamp"])
    df_all = df_all.set_index("timestamp").sort_index()

    home_cols = [col for col in df_all.columns if col.startswith("home_")]

    print(f"Detected {len(home_cols)} homes for {country}.")
    print(home_cols)

    for day_name in days:
        selected_day = dataset_days[country][day_name]
        date = f"{selected_day} 00:00:00"
        forecast_end_date = str(pd.Timestamp(date) + pd.Timedelta(days=1))

        print(f"\n{'='*100}")
        print(f"Running {country} - {day_name}")
        print(f"Forecast start: {date}")
        print(f"Forecast end:   {forecast_end_date}")
        print(f"{'='*100}")

        home_test_preds_list = []

        for home_id in home_cols:
            print(f"\n--- Running home: {home_id} ---")
            df_home_wide = df_all[[home_id] + weather_cols].copy()

            val_start_for_selection = pd.Timestamp(date) - pd.Timedelta(days=3)

            df_home_nf, selected_weather_lag_features = build_feature_enriched_df(
                df_all=df_home_wide,
                home_cols=[home_id],
                weather_cols=weather_cols,
                country=country,
                forecast_horizon=forecast_horizon,
                top_k_per_weather=6,
                max_weather_lag=forecast_horizon * 2,
                train_end_for_selection=val_start_for_selection,
            )

            train_df, val_df, test_df = split_train_val_test_local(
                df_nf=df_home_nf,
                test_start=date,
                forecast_horizon=forecast_horizon,
                training_size=training_size,
                val_days=3,
                freq_minutes=15,
            )

            future_known_candidates = get_feature_groups()

            selected_future_known, selected_historical_lags, importance_df = select_exogenous_features(
                train_df=train_df,
                future_known_features=future_known_candidates,
                historical_lagged_features=selected_weather_lag_features,
                alpha=0.20,
                rf_params={
                    "n_estimators": 500,
                    "random_state": 42,
                    "n_jobs": -1,
                    "max_features": "sqrt",
                },
                fallback_top_k=25,
            )

            train_df = keep_only_required_columns(train_df, selected_future_known, selected_historical_lags)
            val_df = keep_only_required_columns(val_df, selected_future_known, selected_historical_lags)
            test_df = keep_only_required_columns(test_df, selected_future_known, selected_historical_lags)
            
            
            #train_df = train_df.dropna().reset_index(drop=True)
            #val_df = val_df.dropna().reset_index(drop=True)
            #test_df = test_df.dropna().reset_index(drop=True)


            objective_local = make_objective_local(
                train_df=train_df,
                val_df=val_df,
                forecast_horizon=forecast_horizon,
                selected_future_known=selected_future_known,
                selected_historical_lags=selected_historical_lags,
                freq="15min",
            )

            study = optuna.create_study(direction="minimize")
            study.optimize(objective_local, n_trials=opt_trials, show_progress_bar=True)

            best_params = study.best_params

            train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            train_val_df = train_val_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

            final_model = GRU(
                h=forecast_horizon,
                input_size=best_params["input_size"],
                encoder_n_layers=best_params["encoder_n_layers"],
                encoder_hidden_size=best_params["encoder_hidden_size"],
                encoder_dropout=best_params["encoder_dropout"],
                decoder_layers=best_params["decoder_layers"],
                decoder_hidden_size=best_params["decoder_hidden_size"],
                batch_size=best_params["batch_size"],
                learning_rate=best_params["learning_rate"],
                max_steps=MAX_STEPS,
                val_check_steps=min(VAL_CHECK_STEPS, MAX_STEPS),
                scaler_type="standard",
                random_seed=42,
                loss=MSE(),
                futr_exog_list=selected_future_known,
                hist_exog_list=selected_historical_lags,
            )

            nf_final = NeuralForecast(
                models=[final_model],
                freq="15min",
            )

            nf_final.fit(df=train_val_df)

            futr_df_test = test_df[["unique_id", "ds"] + selected_future_known].copy()

            test_preds_df = nf_final.predict(futr_df=futr_df_test)

            test_preds_wide = test_preds_df.pivot(
                index="ds",
                columns="unique_id",
                values="GRU"
            ).sort_index()

            home_test_preds_list.append(test_preds_wide)

        final_test_preds_wide = pd.concat(home_test_preds_list, axis=1).sort_index()




        # ========================================================
        # COUNTRY RUNTIME
        # ========================================================
        country_runtime = time.time() - country_start_time

        print(
            f"\nTotal runtime for {country}: "
            f"{country_runtime:.2f} seconds"
        )

        # ========================================================
        # SAVE / UPDATE JSON WITHOUT OVERWRITING EXISTING CONTENT
        # ========================================================
        json_path = pathlib.Path(project_path) / "Outputs" / f"time_spend_{country}.json"

        if json_path.exists():
            with open(json_path, "r") as f:
                runtime_dict = json.load(f)
        else:
            runtime_dict = {}

        if "Local" not in runtime_dict:
            runtime_dict["Local"] = {}

        runtime_dict["Local"]["GRU"] = country_runtime

        with open(json_path, "w") as f:
            json.dump(runtime_dict, f, indent=4)

        print(f"Saved/updated runtime JSON: {json_path}")




        save_dir = pathlib.Path(project_path) / "Outputs" / "Local models" / "GRU"
        save_dir.mkdir(parents=True, exist_ok=True)

        save_path = save_dir / f"prediction_GRU_{day_name}_{country}.csv"
        final_test_preds_wide.to_csv(save_path, index=True)

        print(f"Saved final_test_preds_wide to: {save_path}")

end_time = time.time()
total_seconds = end_time - start_time
print(f"Total runtime: {total_seconds:.2f} seconds")

# end 